# Safety Gear Detection with a Vision Transformer

Walkthrough of the training pipeline in the `safety_detection` package: load the
*Wearing Safety Gear or Not* images, fine-tune a pretrained ViT-B/16 head, inspect
the results, and try the model in a Gradio demo.

Run this notebook from the repository root (or add it to `sys.path`). The dataset is
expected at `GearedorNot/` with one sub-folder per class.

In [ ]:
# On Colab / Kaggle, uncomment to install the dependencies
# !pip install -q -r requirements.txt

In [ ]:
import torch
from torch import nn

from safety_detection.data import create_dataloaders
from safety_detection.engine import evaluate, fit, plot_history
from safety_detection.model import build_model, load_checkpoint
from safety_detection.predict import Predictor, default_device
from safety_detection.train import set_seed

DATA_DIR = "GearedorNot"           # e.g. "/content/drive/MyDrive/GearedorNot" on Colab
CHECKPOINT = "checkpoints/vit_ppe.pt"
SEED = 42

set_seed(SEED)
device = default_device()
device

## 1. Data
A stratified, seeded 70/30 split. Augmentation (random crop, flip, colour jitter) is applied to the training split only; validation images are just resized and normalised with ImageNet statistics.

In [ ]:
train_loader, val_loader, class_names = create_dataloaders(DATA_DIR, batch_size=32, val_fraction=0.3, seed=SEED)
print(class_names)
print(f"train: {len(train_loader.dataset)} | val: {len(val_loader.dataset)}")

## 2. Model
ViT-B/16 pretrained on ImageNet. The backbone is frozen and only a new 768→2 linear head is trained.

In [ ]:
model = build_model(len(class_names), pretrained=True, freeze_backbone=True).to(device)
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"trainable params: {sum(p.numel() for p in trainable):,} / {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(trainable, lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

## 3. Train
The checkpoint with the best validation accuracy is saved to `checkpoints/vit_ppe.pt`.

In [ ]:
history = fit(model, train_loader, val_loader, optimizer, loss_fn, epochs=10,
              device=device, class_names=class_names, checkpoint_path=CHECKPOINT)
plot_history(history);

## 4. Evaluate the best checkpoint

In [ ]:
best_model, _ = load_checkpoint(CHECKPOINT, device)
result = evaluate(best_model, val_loader, loss_fn, device, len(class_names))
print(f"val accuracy: {result.accuracy:.4f}")
print("confusion matrix (rows = true, cols = predicted):")
for name, row in zip(class_names, result.confusion):
    print(f"  {name:<26} {row}")

## 5. Interactive demo

In [ ]:
from safety_detection.app import build_interface

build_interface(Predictor(CHECKPOINT, device)).launch()